In [68]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import os
import time

### Unstructured

In [2]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(4, 3)
        self.fc2 = nn.Linear(3, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x

In [19]:
model = MyModel()

In [20]:
state_dict = {
    "fc1.weight": torch.tensor([
        [0.1, 0.2, 0.3, 0.1],
        [0.2, 0.3, 0.1, 0.2],
        [0.3, 0.1, 0.2, 0.3],
    ]),

    "fc1.bias": torch.tensor([
        0.1, 0.2, 0.3
    ]),

    "fc2.weight": torch.tensor([
        [0.1, 0.2, 0.3],
        [0.2, 0.3, 0.1],
    ]),

    "fc2.bias": torch.tensor([
        0.1, 0.2
    ]),
}

model.load_state_dict(state_dict)

<All keys matched successfully>

In [21]:
model.fc1.weight_mask

AttributeError: 'Linear' object has no attribute 'weight_mask'

In [22]:
model.fc1.weight_orig

AttributeError: 'Linear' object has no attribute 'weight_orig'

In [23]:
model.fc1.weight

Parameter containing:
tensor([[0.1000, 0.2000, 0.3000, 0.1000],
        [0.2000, 0.3000, 0.1000, 0.2000],
        [0.3000, 0.1000, 0.2000, 0.3000]], requires_grad=True)

In [24]:
prune.l1_unstructured(
    model.fc1,
    name="weight",
    amount=0.5
)

Linear(in_features=4, out_features=3, bias=True)

In [25]:
model.fc1.weight

tensor([[0.0000, 0.0000, 0.3000, 0.0000],
        [0.2000, 0.3000, 0.0000, 0.0000],
        [0.3000, 0.0000, 0.2000, 0.3000]], grad_fn=<MulBackward0>)

In [26]:
model.fc1.weight_mask

tensor([[0., 0., 1., 0.],
        [1., 1., 0., 0.],
        [1., 0., 1., 1.]])

In [27]:
model.fc1.weight_orig

Parameter containing:
tensor([[0.1000, 0.2000, 0.3000, 0.1000],
        [0.2000, 0.3000, 0.1000, 0.2000],
        [0.3000, 0.1000, 0.2000, 0.3000]], requires_grad=True)

### Structured

In [101]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(4, 3)
        self.fc2 = nn.Linear(3, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x

In [102]:
model = MyModel()

state_dict = {
    "fc1.weight": torch.tensor([
        [0.1, 0.2, 0.3, 0.1],
        [0.2, 0.3, 0.1, 0.2],
        [0.3, 0.1, 0.2, 0.3],
    ]),

    "fc1.bias": torch.tensor([
        0.1, 0.2, 0.3
    ]),

    "fc2.weight": torch.tensor([
        [0.1, 0.2, 0.3],
        [0.2, 0.3, 0.1],
    ]),

    "fc2.bias": torch.tensor([
        0.1, 0.2
    ]),
}

model.load_state_dict(state_dict)

<All keys matched successfully>

In [104]:
importance = model.fc1.weight.abs().sum(dim=1) 
importance

tensor([0.7000, 0.8000, 0.9000], grad_fn=<SumBackward1>)

In [105]:
indices = torch.topk(importance, 2).indices
indices

tensor([2, 1])

In [106]:
prune.ln_structured(
    model.fc1,
    name="weight",
    amount=1,
    n=2,
    dim=0
)

Linear(in_features=4, out_features=3, bias=True)

In [109]:
model.fc1.weight

tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.3000, 0.1000, 0.2000],
        [0.3000, 0.1000, 0.2000, 0.3000]], grad_fn=<MulBackward0>)

In [50]:
model.fc1.weight_orig

Parameter containing:
tensor([[0.1000, 0.2000, 0.3000, 0.1000],
        [0.2000, 0.3000, 0.1000, 0.2000],
        [0.3000, 0.1000, 0.2000, 0.3000]], requires_grad=True)

In [108]:
model.fc1.weight_mask

tensor([[0., 0., 0., 0.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

### Big Model

In [78]:
def get_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)

class MyModel(nn.Module):
    def __init__(self, n_layers=3, hidden_dim=1000):
        super().__init__()
        layers = []
        for _ in range(n_layers):
            layers.append(nn.Linear(hidden_dim, hidden_dim, bias=False))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_dim, 1, bias=False))
        self.layers = nn.Sequential(*layers)
    def forward(self, x):
        return self.layers(x)

class PrunedMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers=100):
        super().__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim, bias=False))
        layers.append(nn.ReLU())
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim, bias=False))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_dim, 1, bias=False))
        self.layers = nn.Sequential(*layers)
    def forward(self, x):
        return self.layers(x)

In [79]:
def structured_prune_and_rebuild(model, prune_ratio=0.3):
    old_dim = 1000
    new_dim = int(old_dim * (1 - prune_ratio))
    linear_layers = [l for l in model.layers if isinstance(l, nn.Linear)]
    pruned_model = PrunedMLP(
        input_dim=old_dim,
        hidden_dim=new_dim,
        n_layers=100
    )
    new_linear_layers = [l for l in pruned_model.layers if isinstance(l, nn.Linear)]   
    
    prev_indices = torch.arange(old_dim)
    with torch.no_grad():
        for i, (old_layer, new_layer) in enumerate(
            zip(linear_layers[:-1], new_linear_layers[:-1])
        ):    
            importance = old_layer.weight.abs().sum(dim=1)         
            indices = torch.topk(importance, new_dim).indices
            new_layer.weight.copy_(
                old_layer.weight[indices][:, prev_indices]
            )       
            prev_indices = indices
        old_out = linear_layers[-1]
        new_out = new_linear_layers[-1]
        new_out.weight.copy_(old_out.weight[:, prev_indices])    
    return pruned_model        

In [85]:
model = MyModel(n_layers=100)
with torch.no_grad():
    for param in model.parameters():
        param.uniform_(-0.1, 0.1)
        
torch.save(model.state_dict(), "model.pth")
print(f"Original size : {get_size_mb('model.pth'):.2f} MB")

Original size : 381.50 MB


In [96]:
structured_pruned_model = structured_prune_and_rebuild(model, prune_ratio=0.1)
torch.save(structured_pruned_model.state_dict(), "pruned_model.pth")
print(f"Pruned size   : {get_size_mb('pruned_model.pth'):.2f} MB")

Pruned size   : 309.37 MB


In [97]:
x = torch.rand(1, 1000)

with torch.no_grad():
    y_original = model(x)
    y_pruned = structured_pruned_model(x)

print("Original:")
print(y_original)

print("Pruned:")
print(y_pruned)

Original:
tensor([[-2.8605e+10]])
Pruned:
tensor([[-5.5918e+08]])
